In [1]:
import duckdb

## Set up a database and load the raw data into a table

We're going to create a DuckDB database to persist the data we have extracted from the CSV files and then take it through a few transformation steps to finally generate some useful analytics over the data.

In [2]:
from pathlib import Path
LAND_REGISTRY_DATABASE_PATH = Path("../databases/land_registry.db")

In [3]:
if LAND_REGISTRY_DATABASE_PATH.exists():
    LAND_REGISTRY_DATABASE_PATH.unlink()
else:
    LAND_REGISTRY_DATABASE_PATH.parent.mkdir(parents=True, exist_ok=True)

In [26]:
db = duckdb.connect(LAND_REGISTRY_DATABASE_PATH)
# db = duckdb.connect(":memory:")

In [27]:
db.sql("SELECT * from information_schema.tables")

┌───────────────┬──────────────┬──────────────────────────────────────────┬────────────┬──────────────────────────────┬──────────────────────┬───────────────────────────┬──────────────────────────┬────────────────────────┬────────────────────┬──────────┬───────────────┬───────────────┐
│ table_catalog │ table_schema │                table_name                │ table_type │ self_referencing_column_name │ reference_generation │ user_defined_type_catalog │ user_defined_type_schema │ user_defined_type_name │ is_insertable_into │ is_typed │ commit_action │ TABLE_COMMENT │
│    varchar    │   varchar    │                 varchar                  │  varchar   │           varchar            │       varchar        │          varchar          │         varchar          │        varchar         │      varchar       │ varchar  │    varchar    │    varchar    │
├───────────────┼──────────────┼──────────────────────────────────────────┼────────────┼──────────────────────────────┼────────────────────

## Load raw data into a table

In [6]:
db.sql("CREATE SCHEMA IF NOT EXISTS bronze")

In [7]:
DATA_PATH = Path("../data/land_registry/")

In [8]:
db.sql(
    f"""
    CREATE TABLE IF NOT EXISTS bronze.price_paid_raw AS
    SELECT
    column00 AS 'id',
    column01 AS 'price',
    column02 AS 'date',
    column03 AS 'postcode',
    column04 AS 'property_type',
    column05 AS 'old_new',
    column06 AS 'duration',
    column07 AS 'paon',
    column08 AS 'saon',
    column09 AS 'street',
    column10 AS 'locale',
    column11 AS 'town_city',
    column12 AS 'district',
    column13 AS 'county',
    column14 AS 'ppd_category',
    column15 AS 'record_type',
    FROM read_csv('{DATA_PATH / "pp-*.csv"}');
    """
)

## Add features

In [9]:
db.sql("CREATE SCHEMA IF NOT EXISTS silver")

In [ ]:
db.sql(
    f"""
    CREATE OR REPLACE VIEW silver.price_paid_cleaned AS
    SELECT
    id,
    price,
    date,
    year(date) AS 'year_of_sale',
    date_trunc('month', date) AS 'month_of_sale',
    CASE property_type
          WHEN 'D' THEN 'Detached'
          WHEN 'S' THEN 'Semi-Detached'
          WHEN 'T' THEN 'Terraced'
          WHEN 'F' THEN 'Flat'
          WHEN 'O' THEN 'Other'
          ELSE property_type
    END AS 'property_type',
    CASE old_new
          WHEN 'Y' THEN 'New'
          WHEN 'N' THEN 'Existing'
          ELSE old_new
    END AS 'old_new',
    postcode,
    paon,
    saon,
    street,
    locale,
    town_city,
    district,
    county
    FROM bronze.price_paid_raw;
    """
)

In [11]:
db.sql(
    """
    SELECT
    * 
    FROM silver.price_paid_cleaned LIMIT 10
    """
)

┌────────────────────────────────────────┬────────┬─────────────────────┬──────────────┬─────────────────────┬───────────────┬──────────┬──────────┬───────────────┬────────────┬─────────┬─────────────────┬────────────────┬─────────────┬───────────────────────┬───────────────────────┐
│                   id                   │ price  │        date         │ year_of_sale │    month_of_sale    │ property_type │ old_new  │ postcode │ postcode_area │    paon    │  saon   │     street      │     locale     │  town_city  │       district        │        county         │
│                varchar                 │ int64  │      timestamp      │    int64     │      timestamp      │    varchar    │ varchar  │ varchar  │    varchar    │  varchar   │ varchar │     varchar     │    varchar     │   varchar   │        varchar        │        varchar        │
├────────────────────────────────────────┼────────┼─────────────────────┼──────────────┼─────────────────────┼───────────────┼──────────┼────────

## Projecting insights

In [12]:
db.sql("CREATE SCHEMA IF NOT EXISTS gold")

In [13]:
db.sql(
    f"""
    CREATE OR REPLACE VIEW gold.average_price_by_month_and_property_type AS
    SELECT
    property_type,
    MEDIAN(price) AS median_price,
    month_of_sale,
    FROM silver.price_paid_cleaned
    WHERE property_type <> 'Other'
    GROUP BY ALL
    ORDER BY month_of_sale, property_type;
    """
)

In [14]:
db.sql("FROM gold.average_price_by_month_and_property_type LIMIT 10")

┌───────────────┬──────────────┬─────────────────────┐
│ property_type │ median_price │    month_of_sale    │
│    varchar    │    double    │      timestamp      │
├───────────────┼──────────────┼─────────────────────┤
│ Detached      │     300000.0 │ 2016-01-01 00:00:00 │
│ Flat          │     198000.0 │ 2016-01-01 00:00:00 │
│ Semi-Detached │     185000.0 │ 2016-01-01 00:00:00 │
│ Terraced      │     166995.0 │ 2016-01-01 00:00:00 │
│ Detached      │     299950.0 │ 2016-02-01 00:00:00 │
│ Flat          │     194200.0 │ 2016-02-01 00:00:00 │
│ Semi-Detached │     184995.0 │ 2016-02-01 00:00:00 │
│ Terraced      │     165000.0 │ 2016-02-01 00:00:00 │
│ Detached      │     305000.0 │ 2016-03-01 00:00:00 │
│ Flat          │     192000.0 │ 2016-03-01 00:00:00 │
└───────────────┴──────────────┴─────────────────────┘
  10 rows                                  3 columns

## Inspect contents of database

In [15]:
db.sql("SELECT * from information_schema.tables")

┌───────────────┬──────────────┬──────────────────────────────────────────┬────────────┬──────────────────────────────┬──────────────────────┬───────────────────────────┬──────────────────────────┬────────────────────────┬────────────────────┬──────────┬───────────────┬───────────────┐
│ table_catalog │ table_schema │                table_name                │ table_type │ self_referencing_column_name │ reference_generation │ user_defined_type_catalog │ user_defined_type_schema │ user_defined_type_name │ is_insertable_into │ is_typed │ commit_action │ TABLE_COMMENT │
│    varchar    │   varchar    │                 varchar                  │  varchar   │           varchar            │       varchar        │          varchar          │         varchar          │        varchar         │      varchar       │ varchar  │    varchar    │    varchar    │
├───────────────┼──────────────┼──────────────────────────────────────────┼────────────┼──────────────────────────────┼────────────────────

## Inspect the files

In [16]:
database_files = list(LAND_REGISTRY_DATABASE_PATH.parent.glob("*.db"))
csv_files = list(DATA_PATH.glob("*.csv"))

In [17]:
database_files

[PosixPath('../databases/land_registry.db')]

In [18]:
import polars as pl

files_to_analyse = pl.DataFrame(
    { 
        "file_name": [f.name for f in database_files + csv_files],
        "file_size": [f.stat().st_size for f in database_files + csv_files],
        "file_type": [f.suffix for f in database_files + csv_files]
    }
)

In [19]:
files_to_analyse

file_name,file_size,file_type
str,i64,str
"""land_registry.db""",443297792,""".db"""
"""pp-2021.csv""",223121026,""".csv"""
"""pp-2018.csv""",180162507,""".csv"""
"""pp-2017.csv""",185326793,""".csv"""
"""pp-2025.csv""",139672841,""".csv"""
…,…,…
"""pp-2024.csv""",161058370,""".csv"""
"""pp-2016.csv""",181579035,""".csv"""
"""pp-2020.csv""",156180925,""".csv"""


In [25]:
(
    files_to_analyse
    .group_by("file_type")
    .agg(pl.col("file_size").sum().alias("total_file_size"))
    .with_columns((pl.col("total_file_size") / (1024 * 1024 * 1024)).alias("total_file_size_gb"))
)

file_type,total_file_size,total_file_size_gb
str,i64,f64
""".csv""",1750275243,1.630071
""".db""",443297792,0.412853


In [21]:
# Calculate total sizes for .db and .csv files
db_size = files_to_analyse.filter(pl.col("file_type") == ".db")["file_size"].sum()
csv_size = files_to_analyse.filter(pl.col("file_type") == ".csv")["file_size"].sum()

# Calculate compression percentage
compression_percentage = (db_size / csv_size) * 100
print(f"DuckDB database is {compression_percentage:.1f}% of the original size of the CSV files.")

DuckDB database is 25.3% of the original size of the CSV files.


## Add visualisation over the analytics

Finally we convert the Gold `average_price_by_month_and_property_type` view to a Polars dataframe so we can visualise the data on a line chart using the Plotly Express charting package.

In [22]:
import plotly.express as px

fig = px.line(
    db.sql("FROM gold.average_price_by_month_and_property_type").pl(),
    x="month_of_sale",
    y="median_price",
    color="property_type",
    line_dash="property_type",
    color_discrete_sequence=["#8CC600", "black", "darkgrey", "#EE7423"],
    line_dash_sequence=["solid", "dash", "dot", "dashdot"],
    title="Median Price by Month and Property Type"
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Median Price",
    legend_title="Property Type",
)

fig.show()

In [23]:
# Close DuckDB connection
db.close()